# 06 — Market Evaluation & Full v4 Pipeline

Runs every v4 capability end-to-end:

1. **Market backtest** — model rank IC vs ADP rank IC + long/short residual test (the alpha measurement)
2. **Hybrid model training** — season-grouped CV, per-season z-scoring, EB shrinkage, xTD priors (automatic)
3. **Calibrated uncertainty** — walk-forward residual quantiles + coverage validation
4. **Availability** — expected games replaces the flat 17-game assumption
5. **Season Monte Carlo** — P10/P50/P90 season totals
6. **Rookies** — draft capital + landing spot, merged into the board
7. **Draft board export** — VOR + tiers over the full universe

**First-run warning:** the data cache is empty and `TRAINING_SEASONS` is now 2012–2024, so the
matrix build downloads ~3–4 GB of PBP data (20–40 min). Subsequent runs use the parquet cache.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from config import TRAINING_SEASONS, PROJECTION_SEASON, OUTPUT_DIR

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## Build the v4 feature matrix and YoY pairs

The assembler now also computes expected-TD geometry (`x_*_td_rate`, `*_td_oe`) and
routes/TPRR (participation data 2016–2023, snap-share fallback elsewhere).
`ALL_RATE_TARGET_COLS` includes `games_played`, so the pairs carry `next_games_played`
for the availability model.

In [ ]:
from features.assembler import assemble_feature_matrix, build_yoy_pairs
from models.two_stage import ALL_RATE_TARGET_COLS

fm = assemble_feature_matrix(TRAINING_SEASONS)
pairs = build_yoy_pairs(fm, extra_target_cols=ALL_RATE_TARGET_COLS)

print(f"matrix: {fm.shape}, pairs: {pairs.shape}")
print(f"new v4 columns present: ",
      [c for c in ['x_rec_td_rate', 'rec_td_oe', 'x_rush_td_rate', 'tprr',
                   'route_participation', 'next_games_played'] if c in list(fm.columns) + list(pairs.columns)])

## 1. Market backtest — do we beat ADP?

`ic_edge` = model rank IC − ADP rank IC. `ls_spread` = how much the players the model
likes MORE than the market beat their price, minus the same for players it likes less
(positional-rank units). **Positive and consistent `ls_spread` across seasons is alpha;
anything else means the model is repricing the consensus.**

In [ ]:
from data.adp import load_adp, attach_adp
from models.market import rolling_market_backtest, market_baseline
from models.hybrid import HybridProjectionModel

adp = load_adp(list(range(2015, PROJECTION_SEASON + 1)))

# What does the market achieve on its own? (the bar to clear)
pairs_adp = attach_adp(pairs, adp, season_offset=1)
display(market_baseline(pairs_adp).query("season == 'average'"))

market_result = rolling_market_backtest(
    HybridProjectionModel, pairs, adp,
    test_seasons=[2020, 2021, 2022, 2023],
    age_adjust=False,
)
display(market_result)

## 2. Train the production hybrid + inspect EB shrinkage

Training automatically uses season-grouped walk-forward CV and per-season z-scoring.
The shrinkage table shows the fitted prior strength per metric — `obs_weight_at_n50`
vs `_n150` is the sample-size dependence the old fixed constants couldn't express.

In [ ]:
model = HybridProjectionModel()
model.train(pairs)

display(model._two_stage._eb.shrinkage_table())

## 3. Calibrated uncertainty from walk-forward residuals

Re-trains on an expanding window and collects genuinely out-of-sample residuals,
then validates that the nominal 80% interval actually covers ~80%.

In [ ]:
from models.uncertainty import walk_forward_residuals, ResidualQuantiles

resids = walk_forward_residuals(
    lambda: HybridProjectionModel(age_adjust=False), pairs, min_train_seasons=4
)
rq = ResidualQuantiles().fit(resids)
model.set_uncertainty(rq)

display(rq.coverage_report(resids, lo=0.10, hi=0.90))

## 4. Project 2025 + expected games (availability)

`attach_to_projections` replaces the flat 17-game assumption with model-expected
games from age / workload / durability history.

In [ ]:
from models.availability import AvailabilityModel

input_season = max(TRAINING_SEASONS)
features_now = fm[fm["season"] == input_season].reset_index(drop=True)

proj = model.project(features_now, season=PROJECTION_SEASON)

avail = AvailabilityModel().train(pairs)
proj = avail.attach_to_projections(proj, features_now, target_season=PROJECTION_SEASON)

display(proj.head(15)[["player_name", "position", "team", "projected_fpts_pg",
                        "projected_games", "projected_fpts_season"]])

## 5. Season-total distribution (P10 / P50 / P90)

Monte Carlo composing per-game residual uncertainty with games-played uncertainty.
The P90−P10 gap is the ceiling/floor view a mean-ranked board cannot express.

In [ ]:
from models.uncertainty import simulate_season_totals

proj = simulate_season_totals(
    proj, rq, target_season=PROJECTION_SEASON, games_sd=avail.residual_std_
)

display(proj.head(15)[["player_name", "position", "projected_fpts_season",
                        "season_p10", "season_p50", "season_p90"]])

## 6. Rookie projections, merged into the board

Draft capital + landing spot (prior-season team context and QB quality derived from
the feature matrix). Rookies previously received no projection at all.

In [ ]:
from data.loader import load_draft_picks, load_weekly
from models.rookie import RookieModel, merge_rookie_projections

draft_picks = load_draft_picks()
weekly = load_weekly(list(TRAINING_SEASONS))

# Landing-spot environment frames derived from the feature matrix
team_ctx_cols = [c for c in ["team", "season", "team_pace", "team_pass_rate",
                              "team_offensive_epa"] if c in fm.columns]
team_ctx = fm[team_ctx_cols].drop_duplicates(["team", "season"]) if len(team_ctx_cols) > 2 else None
qb_env = (fm[["team", "season", "qb_epa_per_dropback"]].drop_duplicates(["team", "season"])
          if "qb_epa_per_dropback" in fm.columns else None)

rookie_frame = RookieModel.build_training_frame(
    draft_picks, weekly, team_context=team_ctx, qb_coupling=qb_env,
    end_season=input_season,
)
rookie_model = RookieModel().train(rookie_frame)
rookies = rookie_model.project_class(
    draft_picks, PROJECTION_SEASON, team_context=team_ctx, qb_coupling=qb_env
)
display(rookies.head(10))

board = merge_rookie_projections(proj, rookies)
print(f"board: {len(board)} players ({int(board['rookie'].sum())} rookies)")

## 7. VOR + tiers over the full universe, export

Replacement levels now see veterans AND rookies with availability-adjusted totals.

In [ ]:
from models.vor import calculate_vor
from ranking.ranker import generate_rankings, rankings_table
from ranking.tiers import assign_tiers_all_positions

board = calculate_vor(board, league_size=12)
board = assign_tiers_all_positions(board)
board = generate_rankings(board)

display(rankings_table(board, top_n=30))

out_path = OUTPUT_DIR / f"draft_board_{PROJECTION_SEASON}_v4.csv"
board.to_csv(out_path, index=False)
print(f"exported: {out_path}")

## 8. Position rankings vs market ADP vs model-implied ADP

`predicted_adp` maps the model's overall VORP rank onto the market's ADP ladder
(the ADP of the same-ranked market player), so both columns share units.
`adp_edge = adp - predicted_adp`: **positive = the market lets you draft the player
later than the model values them (value); negative = the market price is rich.**

In [ ]:
import numpy as np

board = board.sort_values("vorp", ascending=False).reset_index(drop=True)
board["model_overall_rank"] = np.arange(1, len(board) + 1)
board = attach_adp(board, load_adp([PROJECTION_SEASON]), season_offset=0,
                   season_col="projected_season") if "adp" not in board.columns else board

ladder = np.sort(load_adp([PROJECTION_SEASON])["adp"].values)
idx = np.minimum(board["model_overall_rank"].values - 1, len(ladder) - 1)
pred = ladder[idx]
overflow = board["model_overall_rank"].values > len(ladder)
board["predicted_adp"] = np.round(np.where(
    overflow, ladder[-1] + (board["model_overall_rank"].values - len(ladder)), pred), 1)
board["adp_edge"] = (board["adp"] - board["predicted_adp"]).round(1)
board["model_pos_rank"] = (board.groupby("position", observed=True)["vorp"]
                           .rank(ascending=False, method="min").astype(int))

for pos in ["QB", "RB", "WR", "TE"]:
    sub = board[board["position"] == pos].nsmallest(12, "model_pos_rank")
    show = sub[["model_pos_rank", "player_name", "team", "projected_fpts_season",
                "projected_games", "adp", "predicted_adp", "adp_edge"]].round(1)
    print(f"\n=== {pos} — model vs market ===")
    print(show.to_string(index=False))

board.to_csv(OUTPUT_DIR / f"rankings_vs_adp_{PROJECTION_SEASON}.csv", index=False)

## Interpreting the market backtest

- `adp_ic` is the market's own skill — typically 0.4–0.6 depending on position/season.
- `ic_edge > 0` means the model out-ranks the market on raw ordering; necessary but not sufficient.
- `ls_spread` is the tradeable claim: among the ~20% of players per position where the model most
  disagrees with ADP, did the "longs" beat their price and the "shorts" miss theirs?
- If `ls_spread` hovers near zero: the model has no residual information over the market — the
  correct draft strategy is ADP with tie-breaks, and the next edge must come from new data
  (props-implied projections, TPRR, coaching changes), not more fitting.